In [ ]:
# [Setup]: 
# Had to install minoconda (Windows): https://www.anaconda.com/download/success
# Had to install Microsoft Visual Code, check Desktop development with C++ in install page: https://visualstudio.microsoft.com/visual-cpp-build-tools/
# Had to install Python extension in VSCodium

# Had to run in Anaconda prompt: 
    # conda create -n rcbplates python=3.11 -y
    # conda activate rcbplates
    # conda install -c conda-forge astropy photutils matplotlib numpy scipy pillow pycairo cairo pkg-config -y
    # pip install daschlab

# Then, in VSCodium:
    # Ctrl + Shift + P -> Python: Select Interpreter -> Python 3.11 (rcbplates)


In [ ]:
# Installs necessary packages. Ensure you have followed the steps above before continuing.

pip install daschlab astropy photutils matplotlib numpy scipy pillow pycairo cairo pkg-config astroquery

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement cairo (from versions: none)

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: C:\Users\dapur\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for cairo


In [ ]:
# Downloads the plates available on R CrB. Prints a statement at the end showing how many plates were available, how many were extracted, and how many were not usable.

# Note: Running this will take a while; decrease value in range(XXX) in for i in range(200): to download less plates/take less time.

from IPython.display import HTML, display

# White overlay behind tqdm text for readability
display(HTML("""
<style>
/* Make ALL output text white */
.output, .output_text, .output_stream, .output_stdout,
.cell-output, .cell-output-print, .cell-output-stdout {
    color: white !important;
}

/* Make all text in outputs white including warnings */
.output * {
    color: white !important;
}

/* Specifically target warning messages */
.output .ansi-yellow-fg, .output .ansi-yellow-foreground,
.warning, .Warning, .warnings {
    color: #FFD700 !important;  /* Golden yellow for warnings */
    text-shadow: 0px 0px 5px rgba(255, 215, 0, 0.3);
}

/* tqdm notebook text styling */
.progress-bar,
.tqdm,
.tqdm div,
.tqdm span,
.tqdm .desc,
.tqdm .bar,
.tqdm .progress,
.tqdm .unit,
.tqdm .rate,
.tqdm .eta,
.tqdm .percentage,
.tqdm .bar_cont {
    color: white !important;
    text-shadow:
        0px 0px 3px rgba(255,255,255,0.9),
        0px 0px 6px rgba(255,255,255,0.6);
}

/* Make ALL tqdm text elements white */
.tqdm * {
    color: white !important;
}

/* optional subtle white "overlay glow" behind labels */
.tqdm .desc,
.tqdm .unit,
.tqdm .rate,
.tqdm .eta {
    background: rgba(0, 0, 0, 0.3);
    padding: 2px 6px;
    border-radius: 4px;
}

/* keep bar visible and set bar color */
.tqdm .bar {
    background-color: #4CAF50 !important;
}

/* keep progress text white */
.tqdm .progress-text {
    color: white !important;
}

/* dark background for better visibility */
.tqdm {
    background: rgba(0, 0, 0, 0.2);
    padding: 4px;
    border-radius: 4px;
}

/* Style for regular print statements */
.output pre, .output code {
    color: white !important;
}
</style>
"""))

from daschlab import open_session
from pathlib import Path
from tqdm.notebook import tqdm

session_dir = Path(
    r"C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN\data\R_CrB"
)

session_dir.mkdir(parents=True, exist_ok=True)

sess = open_session(str(session_dir))

sess.select_target("R CrB")
sess.select_refcat("apass")  # AAVSO Photometric All-Sky Survey

exposures = sess.exposures()
print(f"Total exposures : {len(exposures)}")

cutout_dir = session_dir / "cutouts"

# Download loop
# Every successfully downloaded index is appended to this file immediately
# (and flushed to disk), so if the process is interrupted at any point,
# rerunning the script will skip everything already done.
checkpoint_path = session_dir / "completed_indices.txt"

completed_indices = set()
if checkpoint_path.exists():
    with open(checkpoint_path, "r") as f:
        completed_indices = {
            int(line.strip())
            for line in f
            if line.strip().isdigit()
        }

print(f"Resuming downloads : {len(completed_indices)} cutouts already completed previously.")

remaining_indices = [
    i for i in range(len(exposures))
    if i not in completed_indices
]

print(f"{len(remaining_indices)} cutouts remaining to download.")

# Open in append mode so we never clobber previous progress; "buffering=1" is
# line-buffered so each write is flushed promptly without manual flush calls.
with open(checkpoint_path, "a", buffering=1) as ckpt_f:
    for i in tqdm(
        remaining_indices,
        desc="Downloading cutouts",
        unit="plate"
    ):
        try:
            sess.cutout(i)  # REMOVED the output_dir parameter - it doesn't exist!
            ckpt_f.write(f"{i}\n")
        except Exception as e:
            print(f"\nSkipping exposure {i}: cannot download cutout ({e})")
            continue

# IMPORTANT: now process ONLY actual FITS files
# The cutouts are saved directly in the session directory
cutouts = sorted(session_dir.glob("*.fits"))
print(f"Found {len(cutouts)} cutout files")

# If no cutouts found in session_dir, try the cutout_dir as fallback
if len(cutouts) == 0:
    cutouts = sorted(cutout_dir.glob("*.fits"))
    print(f"Fallback : Found {len(cutouts)} cutout files in cutout_dir")

# Final download summary
total_plates = len(exposures)
downloaded_plates = len(cutouts)
failed_downloads = total_plates - downloaded_plates

print("\n========== FINAL SUMMARY ==========")
print(f"Total plates that exist : {total_plates}")
print(f"Total plates successfully downloaded : {downloaded_plates}")
print(f"Total plates not downloaded (errors) : {failed_downloads}")
print("===================================\n")

Total exposures : 15217
Resuming downloads : 15205 cutouts already completed previously.
12 cutouts remaining to download.


- Querying API ...
- Fetched 1399680 bytes in 6 seconds

Skipping exposure 101: cannot download cutout (Illegal value: masked.)
- Querying API ...
- Fetched 1399680 bytes in 4 seconds

Skipping exposure 281: cannot download cutout (Illegal value: masked.)
- Querying API ...
- Fetched 1399680 bytes in 4 seconds

Skipping exposure 1688: cannot download cutout (Illegal value: masked.)
- Querying API ...
- Fetched 1399680 bytes in 5 seconds

Skipping exposure 7498: cannot download cutout (Illegal value: masked.)
- Querying API ...
- Fetched 1399680 bytes in 5 seconds

Skipping exposure 7499: cannot download cutout (Illegal value: masked.)
- Querying API ...
- Fetched 1399680 bytes in 5 seconds

Skipping exposure 7665: cannot download cutout (Illegal value: masked.)
- Querying API ...
- Fetched 1399680 bytes in 4 seconds

Skipping exposure 7692: cannot download cutout (Illegal value: masked.)
- Querying API ...
- Fetched 1399680 bytes in 4 seconds

Skipping exposure 7693: cannot download cu

In [ ]:
# Plate cutout visualizer. Shows the cutout, the header, and pixel data for all downloaded plates.

from pathlib import Path
from io import BytesIO
from astropy.io import fits
from astropy.time import Time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# Dark styling for the dropdown + toggles + table + kill any default borders/backgrounds/outlines
display(HTML("""
<style>

.dark-dropdown select {
    background-color: #1e1e1e !important;
    color: white !important;
    border: 1px solid #555 !important;
}

.dark-dropdown select option {
    background-color: #1e1e1e !important;
    color: white !important;
}

.dark-toggle {
    color: white !important;
    background-color: #333 !important;
    border: 1px solid #777 !important;
    font-weight: bold !important;
    font-size: 13px !important;
    padding: 6px 10px !important;
}

.dark-toggle:hover {
    background-color: #4a4a4a !important;
}

/* Applied on top of dark-toggle when a toggle is switched on, for visibility */
.dark-toggle-active {
    background-color: #2e7d32 !important;
    border: 1px solid #66bb6a !important;
}

.dark-toggle-active:hover {
    background-color: #388e3c !important;
}

.dark-output,
.dark-output .output,
.dark-output .widget-output,
.jp-OutputArea-output,
.jp-OutputArea-child,
.cell-output-ipywidget-background,
.output,
.output_wrapper,
.widgetarea,
.jp-RenderedHTMLCommon,
.jupyter-widgets {
    background-color: #111 !important;
    border: none !important;
    box-shadow: none !important;
    outline: none !important;
}

body, .jp-Notebook, .jp-WindowedPanel-outer, .jp-Cell-outputArea {
    background-color: #111 !important;
}

.dark-table {
    border-collapse: collapse;
    color: #ddd;
    background-color: #111;
    font-family: monospace;
    font-size: 11px;
}

.dark-table th,
.dark-table td {
    border: 1px solid #333;
    padding: 2px 6px;
    text-align: right;
}

.dark-table th {
    background-color: #222;
    color: white;
}

.table-scroll-box {
    max-height: 500px;
    max-width: 100%;
    overflow: auto;
    background-color: #111;
}

.header-table {
    border-collapse: collapse;
    color: #ddd;
    background-color: #111;
    font-family: monospace;
    font-size: 12px;
    width: 100%;
}

.header-table th,
.header-table td {
    border: 1px solid #333;
    padding: 4px 10px;
    text-align: left;
}

.header-table th {
    background-color: #222;
    color: white;
}

.header-scroll-box {
    max-height: 300px;
    max-width: 100%;
    overflow: auto;
    background-color: #111;
    margin-bottom: 20px;
}

</style>
"""))

cutout_dir = Path(
    r"C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN\data\R_CrB\cutouts"
)

cutouts = sorted(cutout_dir.glob("*.fits"))
print(f"Found {len(cutouts)} cutouts")


# Pull an observation date out of the header (falls back to filename if none found)
def get_plate_date(path):
    try:
        hdr = fits.getheader(path)
    except Exception:
        return path.stem

    for key in ("DATE-OBS", "DATE_OBS", "DATEORIG"):
        if key in hdr and hdr[key]:
            return str(hdr[key])

    for key in ("MJD",):
        if key in hdr and hdr[key]:
            try:
                return Time(float(hdr[key]), format="mjd").iso.split(" ")[0]
            except Exception:
                pass

    for key in ("JD",):
        if key in hdr and hdr[key]:
            try:
                return Time(float(hdr[key]), format="jd").iso.split(" ")[0]
            except Exception:
                pass

    return path.stem


# Build dropdown options as "date - filename"
dropdown_options = []

for f in cutouts:
    filename = str(f.name).strip()

    dropdown_options.append(
        (f"{get_plate_date(f)} - {filename}", f)
    )

dropdown = widgets.Select(
    options=dropdown_options,
    rows=12,
    description="",
    layout=widgets.Layout(width="1200px", height="300px")
)

dropdown.add_class("dark-dropdown")

dropdown_container = widgets.HBox(
    [dropdown],
    layout=widgets.Layout(
        justify_content="center",
        width="100%"
    )
)


# Toggles for header + pixel data
toggle_header = widgets.ToggleButton(
    value=False,
    description="Show Header",
    layout=widgets.Layout(width="160px")
)

toggle_data = widgets.ToggleButton(
    value=False,
    description="Show Pixel Data [SLOW]",
    layout=widgets.Layout(width="180px")
)

toggle_header.add_class("dark-toggle")
toggle_data.add_class("dark-toggle")


def make_label_handler(toggle, label):
    def handler(change):
        if change["new"]:
            toggle.description = f"Hide {label}"
            toggle.add_class("dark-toggle-active")
        else:
            toggle.description = f"Show {label}"
            toggle.remove_class("dark-toggle-active")
    return handler


toggle_header.observe(make_label_handler(toggle_header, "Header"), names="value")
toggle_data.observe(make_label_handler(toggle_data, "Pixel Data"), names="value")


toggles_container = widgets.HBox(
    [toggle_header, toggle_data],
    layout=widgets.Layout(
        justify_content="center",
        width="100%",
        margin="10px 0px"
    )
)


output = widgets.Output(
    layout=widgets.Layout(
        padding="10px",
        width="100%",
        display="flex",
        justify_content="center",
        align_items="center"
    )
)

output.add_class("dark-output")

output_container = widgets.HBox(
    [output],
    layout=widgets.Layout(
        justify_content="center",
        width="100%"
    )
)

headers_output = widgets.Output(layout=widgets.Layout(padding="10px", width="100%"))
headers_output.add_class("dark-output")

headers_container = widgets.HBox(
    [headers_output],
    layout=widgets.Layout(justify_content="center", width="100%")
)

data_output = widgets.Output(layout=widgets.Layout(padding="10px", width="100%"))
data_output.add_class("dark-output")

data_container = widgets.HBox(
    [data_output],
    layout=widgets.Layout(justify_content="center", width="100%")
)

current_path = None
current_data = None

def render_header():
    with headers_output:
        clear_output(wait=True)

        if not toggle_header.value or current_path is None:
            return

        header = fits.getheader(current_path)

        header_df = pd.DataFrame(list(header.items()), columns=['Keyword', 'Value'])

        header_html = header_df.to_html(
            classes="header-table",
            border=0,
            index=False
        )

        display(HTML("<h3 style='color:white;text-align:center;'>FITS Header</h3>"))
        display(HTML(f'<div class="header-scroll-box">{header_html}</div>'))


def render_data():
    with data_output:
        clear_output(wait=True)

        if not toggle_data.value or current_data is None:
            return

        df = pd.DataFrame(current_data)
        table_html = df.to_html(classes="dark-table", border=0)

        display(HTML("<h3 style='color:white;text-align:center;'>Pixel Data</h3>"))
        display(HTML(f'<div class="table-scroll-box">{table_html}</div>'))


def show_plate(change):
    global current_path, current_data

    current_path = change["new"]
    current_data = fits.getdata(current_path)

    with output:
        clear_output(wait=True)

        fig = plt.figure(figsize=(12, 10), facecolor="#111111")

        gs = gridspec.GridSpec(1, 3, width_ratios=[0.05, 1.0, 0.05], wspace=0.03)

        ax_spacer = fig.add_subplot(gs[0])
        ax = fig.add_subplot(gs[1])
        cax = fig.add_subplot(gs[2])

        ax_spacer.axis("off")

        fig.patch.set_facecolor("#111111")
        ax.set_facecolor("#111111")

        im = ax.imshow(current_data, origin="lower", cmap="gray")

        cbar = fig.colorbar(im, cax=cax)
        cbar.set_label("Plate Intensity", color="white", fontsize=14)
        cbar.ax.tick_params(colors="white")

        ax.set_title(current_path.stem, fontsize=28, color="white", pad=16)
        ax.set_xlabel("X Pixel", color="white")
        ax.set_ylabel("Y Pixel", color="white")
        ax.tick_params(colors="white")

        buf = BytesIO()
        fig.savefig(buf, format="png", facecolor=fig.get_facecolor(), bbox_inches="tight")
        plt.close(fig)
        buf.seek(0)

        display(widgets.Image(value=buf.read(), format="png"))

    render_header()
    render_data()


dropdown.observe(show_plate, names="value")
toggle_header.observe(lambda change: render_header(), names="value")
toggle_data.observe(lambda change: render_data(), names="value")


display(
    dropdown_container,
    toggles_container,
    output_container,
    headers_container,
    data_container
)


# Show first plate automatically
if cutouts:
    show_plate({"new": cutouts[0]})

Found 10629 cutouts
